# 18.4 第一名以外的選項有什麼價值？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**老師認爲3比100更接近4，hard label會表達嗎？

硬答案只強調冠軍，soft targets還透露誰像第二名。相關候選的相對概率可能給學生更多結構，但錯誤偏好也會傳過去。

**把直覺寫成數學：**one-hot=[1,0,0]與teacher=[.7,.2,.1]含不同訊息。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
hard = torch.tensor([1.0, 0.0, 0.0])
soft = torch.tensor([0.7, 0.2, 0.1])
print("候選[4,3,100]", hard, soft)
assert torch.allclose(soft.sum(), torch.tensor(1.0))

**如何讀結果：**token概率不等於數值距離的必然編碼；例子只是展示可能的額外訊息。

**只改一件事：**只讓老師對錯誤候選給最高概率。
